In [1]:
import os
import numpy as np
from tqdm import tqdm

In [2]:
DATA_DIR = "./data/raw"
OUT_DIR = "./data/processed"
os.makedirs(OUT_DIR, exist_ok=True)

DELTA_PHASE_A = -0.1080
DELTA_PHASE_B = -0.1280
DELTA_EPS = 1e-6
DOWNSAMPLE = 0

In [3]:
def parse_nso(line: str):
    """
    Parse a global-observable line starting with '# Nso'.

    Example input line:
        # Nso 16344 104277 253668 110490 100028 1.163670 3740

    According to the CDT data format, we:
      - take all numeric values after 'Nso'
      - drop the last two values
      - keep the very last value
    This selects the subset of global observables used in the paper.

    Returns:
        List[float]: selected global observables
    """
    # split line into tokens
    parts = line.split()

    # find the position of the 'Nso' keyword
    idx = parts.index("Nso")

    # take everything after 'Nso'
    after = parts[idx + 1:]

    # keep all but the last two entries, and also keep the final entry
    values = after[:-2] + after[-1:]

    # convert all values to float
    return [float(v) for v in values]


def parse_vto(line: str):
    """
    Parse a local-observable line starting with 'Vto'.

    Example input line:
        Vto t x1 x2 x3 x4 x5 x6

    The first two entries ('Vto' and time index t) are discarded.
    Only the six local geometric observables are returned.

    Returns:
        List[float]: local observables for a single time slice
    """
    # split line into tokens
    parts = line.split()

    # skip 'Vto' and time index, keep x1..x6
    return [float(v) for v in parts[2:]]


def parse_delta_from_filename(filename: str) -> float:
    return float(filename.split("-")[3])*-1


def flatten_sample(sample):
    """
    Convert a single CDT configuration into a flat feature vector.

    The feature vector consists of:
      - global observables (Nso)
      - local observables (Vto), ordered by discrete time slice

    This ordering enforces time-translation symmetry after
    cyclic time-shift augmentation.

    Returns:
        List[float]: 1D feature vector (length = 30)
    """
    features = []

    # add global observables
    features.extend(sample["Nso"])

    # add local observables in time order
    for vto_t in sample["Vto"]:
        features.extend(vto_t)

    return features

def label_from_delta(delta):
    """
    Assign a phase label based on delta.

    Deep inside phase A:
        label = 0
    Deep inside phase B:
        label = 1
    Intermediate delta values:
        label = None (not used for training)

    A small tolerance is used to avoid floating-point issues.

    Returns:
        int or None
    """
    if abs(delta - DELTA_PHASE_A) < DELTA_EPS:
        return 0
    if abs(delta - DELTA_PHASE_B) < DELTA_EPS:
        return 1
    return None

In [4]:
files = sorted(
    f for f in os.listdir(DATA_DIR)
    if f.startswith("vto-") and f.endswith(".out")
)

In [5]:
files

['vto-4.80--0.1080-4-100k-torus.out',
 'vto-4.80--0.1100-4-100k-torus.out',
 'vto-4.80--0.1120-4-100k-torus.out',
 'vto-4.80--0.1140-4-100k-torus.out',
 'vto-4.80--0.1160-4-100k-torus.out',
 'vto-4.80--0.1180-4-100k-torus.out',
 'vto-4.80--0.1200-4-100k-torus.out',
 'vto-4.80--0.1220-4-100k-torus.out',
 'vto-4.80--0.1240-4-100k-torus.out',
 'vto-4.80--0.1260-4-100k-torus.out',
 'vto-4.80--0.1280-4-100k-torus.out']

In [6]:
def process_sample(
    current_sample,
    sample_counter,
    skip_samples,
    file_delta,
    file_label,
):
    if sample_counter <= skip_samples:
        return

    assert len(current_sample["Vto"]) == 4
    assert current_sample["Nso"] is not None
    
    # X_shifts[shift].append(flatten_sample(perm))
    # y_shifts[shift].append(file_label)
    X_full.append(flatten_sample(current_sample))
    Delta_full.append(file_delta)
    y_full.append(file_label)


In [7]:
# Containers for the final dataset (filled after concatenation)
X_full = []
y_full = []
Delta_full = []

# Separate buffers for each time-shift variant
# # shift = 0, 1, 2, 3 correspond to cyclic time translations
# X_shifts = [[], [], [], []]
# y_shifts = [[], [], [], []]
# Delta_shifts = [[], [], [], []]


# Loop over all CDT output files (each file corresponds to a fixed delta)
for file_idx, filename in enumerate(tqdm(files, desc="Parsing files")):

    # Number of initial Monte Carlo configurations to discard
    # (thermalization cut; endpoints use a slightly smaller cut)
    # SKIP_SAMPLES = 2000 if file_idx in (0, len(files) - 1) else 2200
    SKIP_SAMPLES = 0
    # Extract delta value from filename and assign phase label (if deep A or B)
    file_delta = parse_delta_from_filename(filename)
    file_label = label_from_delta(file_delta)

    file_path = os.path.join(DATA_DIR, filename)

    # Temporary storage for the currently parsed configuration
    current_sample = None
    sample_counter = 0

    # Read the file line by line
    with open(file_path, "r") as f:
        for line in f:
            line = line.strip()

            # Marker for the beginning of a new configuration
            if line.startswith("# ntime"):

                # If a previous configuration was fully read, process it
                if current_sample is not None:
                            sample_counter += 1
                        
                            process_sample(
                                current_sample,
                                sample_counter,
                                SKIP_SAMPLES,
                                file_delta,
                                file_label,
                            )
                # Initialize a new configuration container
                current_sample = {
                    "Nso": None,   # global observables
                    "Vto": []      # list of local observables (one per time slice)
                }

            # Parse global observables
            elif line.startswith("# Nso"):
                current_sample["Nso"] = parse_nso(line)

            # Parse local observables for a single time slice
            elif line.startswith("Vto"):
                current_sample["Vto"].append(parse_vto(line))


    if current_sample is not None:
                sample_counter += 1
            
                process_sample(
                    current_sample,
                    sample_counter,
                    SKIP_SAMPLES,
                    file_delta,
                    file_label,
                )
    # Report number of equilibrated configurations processed in this file
    print(f"{filename}: parsed {sample_counter - SKIP_SAMPLES} samples")

    
X_full = np.asarray(X_full)
y_full = np.asarray(y_full, dtype=object)
Delta_full = np.asarray(Delta_full)



Parsing files:   9%|██▋                          | 1/11 [00:00<00:03,  2.83it/s]

vto-4.80--0.1080-4-100k-torus.out: parsed 30001 samples


Parsing files:  18%|█████▎                       | 2/11 [00:00<00:03,  2.94it/s]

vto-4.80--0.1100-4-100k-torus.out: parsed 30001 samples


Parsing files:  27%|███████▉                     | 3/11 [00:01<00:02,  2.99it/s]

...

vto-4.80--0.1240-4-100k-torus.out: parsed 60000 samples


Parsing files:  91%|█████████████████████████▍  | 10/11 [00:06<00:00,  1.01it/s]

vto-4.80--0.1260-4-100k-torus.out: parsed 100000 samples


Parsing files: 100%|████████████████████████████| 11/11 [00:08<00:00,  1.34it/s]

vto-4.80--0.1280-4-100k-torus.out: parsed 100000 samples


In [8]:
assert X_full.shape[1] == 30

mask_train = (
    (Delta_full == DELTA_PHASE_A) |
    (Delta_full == DELTA_PHASE_B)
) & (y_full != None)

X_train_aug = [[], [], [], []]
y_train_aug = [[], [], [], []]
Delta_train_aug = [[], [], [], []]

for x, y, delta in zip(
    X_full[mask_train],
    y_full[mask_train],
    Delta_full[mask_train]
):
    nso = x[:6]
    vto = x[6:].reshape(4, 6)

    for shift in range(4):
        vto_shift = np.roll(
            vto,
            -shift,
            axis=0
        )

        x_shift = np.concatenate([
            nso,
            vto_shift.flatten()
        ])

        X_train_aug[shift].append(x_shift)
        y_train_aug[shift].append(y)
        Delta_train_aug[shift].append(delta)


X_train_aug = np.concatenate(
    [np.asarray(X_train_aug[s]) for s in range(4)],
    axis=0
)

y_train_aug = np.concatenate(
    [np.asarray(y_train_aug[s]) for s in range(4)],
    axis=0
)

Delta_train_aug = np.concatenate(
    [np.asarray(Delta_train_aug[s]) for s in range(4)],
    axis=0
)

        

assert len(X_train_aug) == 4 * np.sum(mask_train)
print("Full samples:", len(X_full))
print("Train samples:", np.sum(mask_train))
print("Train augmented:", len(X_train_aug))


Full samples: 570004
Train samples: 130001
Train augmented: 520004


In [9]:
X_full = np.asarray(X_full, dtype=np.int64)

X_train_aug = np.asarray(X_train_aug, dtype=np.int64)
y_train_aug = np.asarray(y_train_aug)
Delta_train_aug = np.asarray(Delta_train_aug)

np.savez(
    os.path.join(OUT_DIR, "FULL_30.npz"),
    X=X_full,
    y=y_full,
    Delta=Delta_full,
)

print("FULL_30.npz zapisany")
print("X_full shape:", X_full.shape)

np.savez(
    os.path.join(OUT_DIR, "TRAIN_30.npz"),
    X=X_train_aug,
    y=y_train_aug,
    Delta=Delta_train_aug,
)

print("TRAIN_30.npz zapisany")
print("X_train shape:", X_train_aug.shape)

FULL_30.npz zapisany
X_full shape: (570004, 30)
TRAIN_30.npz zapisany
X_train shape: (520004, 30)
